# CSI4142 Assignment 2

1. Andrew Pham - 300226985
2. Kevin Yao - 300295024

# Part 1 - Validity Checker

### Dataset

* Name: NYC Property Sales
* Author: City of New York
* Purpose: A record of every building or unit sold in NYC over a period of 1 year.
* Shape: 84548 rows, 22 columns

https://www.kaggle.com/datasets/new-york-city/nyc-property-sales

In [5]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Download latest version
file_path = "nyc-rolling-sales.csv"
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, "new-york-city/nyc-property-sales", path=file_path
)
orig_df = df.copy()

In [ ]:
df.head()

In [ ]:
df.describe()

### Validity Check 1: Data Type Errors

**Type of error:** Data type inconsistency

**Description:** The `BOROUGH` column should be numeric (int), but some values are stored as strings. This breaks numeric validation and downstream analysis.

The error was introduced by randomly selecting 5% of rows in `BOROUGH` and converting their numeric values to strings (e.g., `3` → `'3'`).

In the real world, this type of error could be introduced through human error, or through errors in data collection pipelines such as migrations from previous string names to number codes.

In [ ]:
# Introduce data type errors in BOROUGH (~5% as strings with quotes)
np.random.seed(1)
err_df = df.copy()
err_idx = err_df.sample(frac=0.05, random_state=1).index

# change type to object so we can mix ints and strings
err_df.BOROUGH = err_df.BOROUGH.astype(str)

# Convert sampled rows to strings with single quotes (e.g., 1 -> '1')
err_df.loc[err_idx, "BOROUGH"] = (
    err_df.loc[err_idx, "BOROUGH"].astype(str).map(lambda v: f"'{v}'")
)

# Show a few corrupted values
err_df.loc[err_idx, ["BOROUGH"]].head()

In [ ]:
def fix_borough_dtype(input_df):
    """
    Detect and correct data type errors in the 'BOROUGH' column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'BOROUGH' column.

    Returns:
        output_df (pd.DataFrame): The corrected DataFrame.
        non_numeric_mask (pd.Series): A boolean mask of rows that were corrected.
    """
    output_df = input_df.copy()

    # Normalize by stripping single quotes from strings
    cleaned = output_df["BOROUGH"].astype(str).str.strip("'")

    # Detect non-numeric entries (strings, mixed types)
    non_numeric_mask = pd.to_numeric(cleaned, errors="coerce").isna()

    # Correct by coercing to numeric and restoring to int
    output_df["BOROUGH"] = pd.to_numeric(cleaned, errors="coerce").astype(int)
    return output_df, non_numeric_mask


# Run detection + correction
fixed_df, bad_mask = fix_borough_dtype(err_df)

In [ ]:
# show 3 examples of bad values
print("Bad values before correction:")
display(err_df.loc[err_idx, ["BOROUGH"]].head(3))
print(f"BOROUGH dtype before correction: {err_df['BOROUGH'].dtype}")
# show 3 examples of corrected values
print("Corrected values after correction:")
display(fixed_df.loc[err_idx, ["BOROUGH"]].head(3))
print(f"BOROUGH dtype after correction: {fixed_df['BOROUGH'].dtype}")

**Results:** The function will correctly coerce string representations of numbers into integers, and the whole column will be set to the correct int64 dtype.

### Validity Check 2: Range Errors

**Type of error:** Out-of-range values

**Description:** The `YEAR BUILT` column should be between 1652 ([oldest NYC house](https://www.nypap.org/preservation-history/wyckoff-house/)) and 2026 (current year). Values outside this range are invalid. In fact, there are 6970 rows already in the dataset that have a year built of 0. Presumably, this is because it was unknown or not collected.

In [ ]:
# count the number of rows that are out of range
df["YEAR BUILT"].value_counts()

**How the error was introduced:** Randomly selected ~5% of rows in `YEAR BUILT` and replaced them with out-of-range years (some too old, some in the future), randomly selected within 100 years before or after the range limits.

In [ ]:
# Introduce range errors in YEAR BUILT (~5% out of range)
np.random.seed(2)
range_err_df = df.copy()
range_err_idx = range_err_df.sample(frac=0.05, random_state=2).index

# Split into too-old and future years
half = len(range_err_idx) // 2
old_idx = range_err_idx[:half]
future_idx = range_err_idx[half:]

old_age_possibilities = range(1552, 1652)
future_age_possibilities = range(2026, 2126)

# sample for each row

old_samples = np.random.choice(old_age_possibilities, size=len(old_idx))
future_samples = np.random.choice(future_age_possibilities, size=len(future_idx))

range_err_df.loc[old_idx, "YEAR BUILT"] = old_samples
range_err_df.loc[future_idx, "YEAR BUILT"] = future_samples

display(range_err_df.loc[old_idx][["YEAR BUILT"]].head(3))
display(range_err_df.loc[future_idx][["YEAR BUILT"]].head(3))

In [ ]:
def find_erroneous_year_built_range(input_df, min_year=1652, max_year=2026):
    """
    Detect and return indices of out-of-range years in the 'YEAR BUILT' column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'YEAR BUILT' column.
        min_year (int): The minimum valid year (default: 1652).
        max_year (int): The maximum valid year (default: 2026).

    Returns:
        out_of_range (pd.Series): A boolean mask of rows that were corrected.
    """
    output_df = input_df.copy()
    year_series = pd.to_numeric(output_df["YEAR BUILT"], errors="coerce")
    out_of_range = (year_series < min_year) | (year_series > max_year)

    return out_of_range


# Run detection + correction
out_of_range_mask = find_erroneous_year_built_range(range_err_df)
fixed_range_df = range_err_df[~out_of_range_mask]


print(f"Rows out of range detected: {out_of_range_mask.sum()}")
print("Bad values before correction:")
display(range_err_df.loc[out_of_range_mask, ["YEAR BUILT"]].head(3))
print("\nValue counts after filtering:")
fixed_range_df[["YEAR BUILT"]].describe()

**Qualitative results:** The detector flags any `YEAR BUILT` values outside 1652–2026 and drops those rows because the true year is unknown. In the sample output, invalid years (too old or too new) are identified and removed; the value counts after filtering show only valid years remain.

### Validity Check 3: Format Errors

**Type of error:** Invalid string format

**Description:** The `BUILDING CLASS AT TIME OF SALE` field should be exactly 2 characters: the first must be a capital letter, and the second must be either a single digit or another capital letter (pattern `^[A-Z][0-9A-Z]$`).

**How the error was introduced:** Randomly selected ~5% of rows and replaced `BUILDING CLASS AT TIME OF SALE` with invalid formats (e.g., lowercase, too long, or wrong character order).

In [ ]:
# Introduce format errors in BUILDING CLASS AT TIME OF SALE (~5%)
np.random.seed(3)
format_err_df = df.copy()
format_err_idx = format_err_df.sample(frac=0.05, random_state=3).index

digits = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
letters = [chr(i) for i in range(65, 91)]
letters_lower = [chr(i) for i in range(97, 123)]


def create_invalid_format_lowercase_letter_2_char():
    chr1 = np.random.choice(letters_lower)
    chr2 = np.random.choice(letters)
    return chr1 + chr2


def create_invalid_format_lowercase_letter_1_char_1_digit():
    chr1 = np.random.choice(letters_lower)
    chr2 = np.random.choice(digits)
    return chr1 + chr2


def create_invalid_format_lowercase_letter_1_digit_1_char():
    chr1 = np.random.choice(digits)
    chr2 = np.random.choice(letters)
    return chr1 + chr2


format_err_fns = [
    create_invalid_format_lowercase_letter_2_char,
    create_invalid_format_lowercase_letter_1_char_1_digit,
    create_invalid_format_lowercase_letter_1_digit_1_char,
]

# randomly select one of the functions for each format_err_idx row and generate the invalid format
for err_idx in format_err_idx:
    invalid_format_fn = np.random.choice(format_err_fns)
    invalid_format = invalid_format_fn()
    format_err_df.loc[err_idx, "BUILDING CLASS AT TIME OF SALE"] = invalid_format

print("Example of bad values:")
format_err_df.loc[format_err_idx, ["BUILDING CLASS AT TIME OF SALE"]].head(3)

In [ ]:
def find_erroneous_building_class_format(input_df):
    """
    Detect and return indices of invalid building class formats in the 'BUILDING CLASS AT TIME OF SALE' column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'BUILDING CLASS AT TIME OF SALE' column.

    Returns:
        out_of_range (pd.Series): A boolean mask of rows that were corrected.
    """
    output_df = input_df.copy()
    cleaned = (
        output_df["BUILDING CLASS AT TIME OF SALE"].astype(str).str.strip().str.upper()
    )

    valid_mask = cleaned.str.match(r"^[A-Z][0-9A-Z]$")

    # Correct by normalizing valid values and setting invalids to NaN
    output_df["BUILDING CLASS AT TIME OF SALE"] = cleaned.where(valid_mask, np.nan)
    return ~valid_mask


# Run detection + correction
format_mask = find_erroneous_building_class_format(format_err_df)
fixed_format_df = format_err_df[~format_mask]

print(f"Rows with invalid format detected: {format_mask.sum()}")
print("Bad values before correction:")
display(format_err_df.loc[format_mask, ["BUILDING CLASS AT TIME OF SALE"]].head(3))
print("After correction:")
display(fixed_format_df.loc[format_mask, ["BUILDING CLASS AT TIME OF SALE"]].head(3))

**Qualitative results:** The detector flags any value not matching the pattern `^[A-Z][0-9A-Z]$` (NOT a uppercase letter followed by a digit or other upper case letter). The sample output shows invalid entries (e.g., lowercase or wrong length) and that they are dropped if invalid.

### Validity Check 4: Consistency Errors

**Type of error:** Inconsistent totals

**Description:** For each row, `RESIDENTIAL UNITS + COMMERCIAL UNITS` should equal `TOTAL UNITS`. Any mismatch indicates an inconsistency in the record.

**How the error was introduced:** Randomly selected ~5% of rows and altered `TOTAL UNITS` so it no longer equals `RESIDENTIAL UNITS + COMMERCIAL UNITS`.

In [ ]:
# Introduce consistency errors in unit totals (~5%)
np.random.seed(4)
consistency_df = df.copy()
consistency_idx = consistency_df.sample(frac=0.05, random_state=4).index

# Break the consistency by subtracting 1 from TOTAL UNITS
consistency_df.loc[consistency_idx, "TOTAL UNITS"] = (
    consistency_df.loc[consistency_idx, "TOTAL UNITS"] - 1
)

# show 3 examples of bad values
print("Bad values before correction:")
display(
    consistency_df.loc[
        consistency_idx, ["RESIDENTIAL UNITS", "COMMERCIAL UNITS", "TOTAL UNITS"]
    ].head(3)
)

In [ ]:
# Function to detect and correct consistency errors
def find_erroneous_unit_consistency(input_df):
    """
    Detect and return indices of inconsistent unit totals in the 'TOTAL UNITS' column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'TOTAL UNITS' column.

    Returns:
        inconsistency_mask (pd.Series): A boolean mask of rows that were not consistent.
    """
    output_df = input_df.copy()
    expected_total = output_df["RESIDENTIAL UNITS"] + output_df["COMMERCIAL UNITS"]
    inconsistency_mask = output_df["TOTAL UNITS"] != expected_total

    return inconsistency_mask


# Run detection + correction
inconsistency_mask = find_erroneous_unit_consistency(consistency_df)
fixed_consistency_df = consistency_df[~inconsistency_mask]

print(f"Rows with inconsistent totals detected: {inconsistency_mask.sum()}")
print("Bad values before correction:")
display(
    consistency_df.loc[
        inconsistency_mask,
        ["RESIDENTIAL UNITS", "COMMERCIAL UNITS", "TOTAL UNITS"],
    ].head(3)
)
print("After correction:")
display(
    fixed_consistency_df.loc[
        inconsistency_mask,
        ["RESIDENTIAL UNITS", "COMMERCIAL UNITS", "TOTAL UNITS"],
    ].head(3)
)

**Qualitative results:** The detector flags rows where the computed total (`RESIDENTIAL UNITS + COMMERCIAL UNITS`) does not match `TOTAL UNITS`. The examples show mismatched totals before correction, then we drop as we cannot be sure which column is incorrect.

### Validity Check 5: Uniqueness Errors

**Type of error:** Duplicate records

**Description:** The combination of `ADDRESS` and `SALE DATE` should be unique because it is extremely unlikely that the same property is sold twice on the same day.

**How the error was introduced:** Randomly selected ~5% of rows and duplicated them, creating repeated `ADDRESS` + `SALE DATE` combinations.

In [ ]:
# Introduce uniqueness errors by duplicating ~5% of rows
np.random.seed(5)
unique_err_df = df.copy()
dup_sample = unique_err_df.sample(frac=0.05, random_state=5)
unique_err_df = pd.concat([unique_err_df, dup_sample], ignore_index=True)

print(f"Rows before duplication: {len(df)}")
print(f"Rows after duplication: {len(unique_err_df)}")

In [ ]:
def fix_address_sale_date_uniqueness(input_df):
    """
    Detect and return indices of duplicate `ADDRESS` + `SALE DATE` pairs.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'ADDRESS' and 'SALE DATE' columns.

    Returns:
        output_df (pd.DataFrame): The corrected DataFrame.
        dup_mask (pd.Series): A boolean mask of rows that were duplicated.
    """
    output_df = input_df.copy()
    dup_mask = output_df.duplicated(subset=["ADDRESS", "SALE DATE"], keep=False)

    # Correct by keeping the first occurrence of each duplicate pair
    output_df = output_df.drop_duplicates(subset=["ADDRESS", "SALE DATE"], keep="first")
    return output_df, dup_mask


# Run detection + correction
fixed_unique_df, dup_mask = fix_address_sale_date_uniqueness(unique_err_df)

print("Before correction:")
print(f"Duplicate rows detected: {dup_mask.sum()}")
print("Example duplicates:")
display(unique_err_df.loc[dup_mask, ["ADDRESS", "SALE DATE"]].head(3))
print(
    f"Column combo is unique: {(unique_err_df['ADDRESS'] + unique_err_df['SALE DATE']).is_unique}"
)
print("--------------------------------")
print("After correction:")
display(fixed_unique_df.loc[:, ["ADDRESS", "SALE DATE"]].head(3))
print(
    f"Column combo is unique: {(fixed_unique_df['ADDRESS'] + fixed_unique_df['SALE DATE']).is_unique}"
)

**Qualitative results:** The detector flags duplicate `ADDRESS` + `SALE DATE` pairs. The example output shows repeated combinations before correction, and after dropping duplicates only one record per property per day remains.

### Validity Check 6: Presence Errors

**Type of error:** Missing values

**Description:** The `ZIP CODE` field should be present for each record. Missing values indicate incomplete data.

**How the error was introduced:** Randomly selected ~5% of rows and removed `ZIP CODE` values (set to missing).

In [ ]:
# Introduce presence errors in ZIP CODE (~5% missing)
np.random.seed(6)
presence_df = df.copy()
presence_idx = presence_df.sample(frac=0.05, random_state=6).index

presence_df.loc[presence_idx, "ZIP CODE"] = np.nan

print("Example of missing ZIP CODE values:")
display(presence_df.loc[presence_idx, ["ZIP CODE"]].head(3))

In [ ]:
# Function to detect and correct presence errors in ZIP CODE
def fix_zipcode_presence(input_df):
    output_df = input_df.copy()
    missing_mask = output_df["ZIP CODE"].isna()

    # Correct by dropping rows with missing ZIP CODE
    output_df = output_df.loc[~missing_mask].copy()
    return output_df, missing_mask


# Run detection + correction
fixed_presence_df, missing_mask = fix_zipcode_presence(presence_df)

print(f"Rows with missing ZIP CODE detected: {missing_mask.sum()}")
print("Missing ZIP CODE examples:")
display(presence_df.loc[missing_mask, ["ZIP CODE"]].head(3))
print(f"After correction, removed {missing_mask.sum()} rows with missing ZIP CODE")

**Qualitative results:** The detector flags rows where `ZIP CODE` is missing. The example output shows missing entries before correction, and after dropping those rows the dataset contains only records with a valid ZIP code. Theoretically, this could be re-added based on the address, but that would require some external library to get address data.

### Validity Check 7: Length Errors

**Type of error:** Incorrect string length

**Description:** The `SALE DATE` field should have the same length as `2017-07-19 00:00:00` (19 characters). Any other length indicates a formatting issue.

**How the error was introduced:** Randomly selected ~5% of rows and modified `SALE DATE` strings to be too short or too long.

In [ ]:
# Introduce length errors in SALE DATE (~5% with wrong length)
np.random.seed(7)
length_df = df.copy()
length_idx = length_df.sample(frac=0.05, random_state=7).index

# Make half too short, half too long
half = len(length_idx) // 2
short_idx = length_idx[:half]
long_idx = length_idx[half:]

length_df.loc[short_idx, "SALE DATE"] = (
    length_df.loc[short_idx, "SALE DATE"].astype(str).str.slice(0, 10)
)
length_df.loc[long_idx, "SALE DATE"] = (
    length_df.loc[long_idx, "SALE DATE"].astype(str) + "UTC-5"
)

print("Example of wrong-length SALE DATE values:")
display(length_df.loc[length_idx, ["SALE DATE"]].head(3))

In [ ]:
def find_erroneous_sale_date_length(input_df, expected_len=19):
    """
    Detect and return indices of incorrect string length in the 'SALE DATE' column.

    Args:
        input_df (pd.DataFrame): The input DataFrame containing the 'SALE DATE' column.
        expected_len (int): The expected length of the 'SALE DATE' string.

    Returns:
        wrong_len_mask (pd.Series): A boolean mask of rows that were incorrect.
    """
    output_df = input_df.copy()
    sale_str = output_df["SALE DATE"].astype(str)
    wrong_len_mask = sale_str.str.len() != expected_len

    return wrong_len_mask


# Run detection + correction
wrong_len_mask = find_erroneous_sale_date_length(length_df)
fixed_length_df = length_df[~wrong_len_mask]

print(f"Rows with wrong-length SALE DATE detected: {wrong_len_mask.sum()}")
print("Bad values before correction:")
display(length_df.loc[wrong_len_mask, ["SALE DATE"]].head(3))
print(
    f"After correction, removed {wrong_len_mask.sum()} rows with wrong-length SALE DATE"
)

**Qualitative results:** The detector flags `SALE DATE` strings that are not exactly 19 characters long (format 2017-07-19 00:00:00). The examples show truncated or extended values before correction, and after filtering only correctly formatted sale dates remain.

### Validity Check 8: Look-up Errors

**Type of error:** Invalid categorical value

**Description:** The `BUILDING CLASS CATEGORY` field should be one of the acceptable categories found in the dataset. Any value not in the allowed list is invalid.

**How the error was introduced:** Randomly selected ~5% of rows and replaced `BUILDING CLASS CATEGORY` with incorrect category names that are not in the acceptable list.

In [ ]:
# Introduce look-up errors in BUILDING CLASS CATEGORY (~5%)
np.random.seed(8)
lookup_df = df.copy()
lookup_idx = lookup_df.sample(frac=0.05, random_state=8).index

# we pre-compute the list of acceptable values here based on the dataset,
# but in real life we would have to set this before aggregating the data
acceptable_values = list(df["BUILDING CLASS CATEGORY"].unique())

invalid_categories = [
    "99 UNKNOWN CATEGORY",
    "XX INVALID CLASS",
    "FAKE CATEGORY",
    "MISCELLANEOUS GROUP",
    "NOT A REAL CLASS",
]
lookup_df.loc[lookup_idx, "BUILDING CLASS CATEGORY"] = np.random.choice(
    invalid_categories, size=len(lookup_idx)
)

print("Example of invalid categories:")
display(lookup_df.loc[lookup_idx, ["BUILDING CLASS CATEGORY"]].head(3))

In [ ]:
def find_erroneous_building_class_lookup(input_df, acceptable_values):
    output_df = input_df.copy()
    invalid_mask = ~output_df["BUILDING CLASS CATEGORY"].isin(acceptable_values)

    return invalid_mask


# Run detection + correction
acceptable_values = df["BUILDING CLASS CATEGORY"].unique()
invalid_mask = find_erroneous_building_class_lookup(lookup_df, acceptable_values)

print(f"Rows with invalid categories detected: {invalid_mask.sum()}")
print("Bad values before correction:")
display(lookup_df.loc[invalid_mask, ["BUILDING CLASS CATEGORY"]].head(3))
print(f"After correction, removed {invalid_mask.sum()} rows with invalid categories")

**Qualitative results:** The detector flags any `BUILDING CLASS CATEGORY` not in the acceptable list. The examples show invalid category names before correction, and after filtering only valid categories remain. In a real application, this would be set with a look up/dropdown instead of allowing people to type.

# Part 2 - Missing Data and Imputation

### Dataset

* Name: Laptop Price
* Author: Muhammet Varli
* Purpose: Sample set of Laptop specifications and prices in 2021
* Source: https://www.kaggle.com/datasets/muhammetvarl/laptop-price

In [15]:
# download and load dataset
import os
import pandas as pd
import kagglehub

# get dataset folder
dataset_ref = "muhammetvarl/laptop-price"
dataset_dir = kagglehub.dataset_download(dataset_ref)

# pick the main csv file
csv_files = [f for f in os.listdir(dataset_dir) if f.lower().endswith(".csv")]
file_path = max(csv_files, key=lambda f: os.path.getsize(os.path.join(dataset_dir, f)))
full_path = os.path.join(dataset_dir, file_path)

# load csv with automatic separator detection
df = pd.read_csv(full_path, encoding="latin1", sep=None, engine="python")

# quick check
print(f"loaded file: {file_path}")
print(f"shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

loaded file: laptop_price.csv
shape: 1303 rows x 13 columns


,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


In [7]:
# copy dataset for part 2
laptop_df = df.copy()

# clean column names
laptop_df.columns = laptop_df.columns.str.strip()

# convert ram from text to number
laptop_df["Ram"] = pd.to_numeric(
    laptop_df["Ram"].astype(str).str.replace("GB", "", regex=False),
    errors="coerce"
)

# convert weight from text to number
laptop_df["Weight"] = pd.to_numeric(
    laptop_df["Weight"].astype(str).str.replace("kg", "", regex=False),
    errors="coerce"
)

# convert screen size to number
laptop_df["Inches"] = pd.to_numeric(laptop_df["Inches"], errors="coerce")

# keep price column numeric
price_col = "Price_euros"
laptop_df[price_col] = pd.to_numeric(laptop_df[price_col], errors="coerce")

# show current missing counts
laptop_df[["Ram", "Weight", "Inches", price_col]].isna().sum()

Ram            0
Weight         0
Inches         0
Price_euros    0
dtype: int64

### Imputation Test 1 - Default Value Imputation (Univariate)

I use **default value imputation** on `Inches`.
I replace each missing value with one default number: the median screen size.

**How missing data is introduced (MCAR):**
- Column: `Inches`
- Method: randomly remove 10% of available values.
- Missingness type: **MCAR** (Missing Completely At Random), because values are removed randomly.

In [8]:
# test 1: mcar + default value imputation on inches
np.random.seed(111)

# copy data
test1_df = laptop_df.copy()
test1_col = "Inches"

# pick 10% rows and store true values
test1_candidates = test1_df.index[test1_df[test1_col].notna()]
test1_remove_idx = test1_df.loc[test1_candidates].sample(frac=0.10, random_state=111).index
test1_true_values = test1_df.loc[test1_remove_idx, test1_col].copy()

# insert missing values
test1_df.loc[test1_remove_idx, test1_col] = np.nan
missing_before_test1 = test1_df[test1_col].isna().sum()

# fill missing values with one default value
default_inches = test1_df[test1_col].median()
test1_df[test1_col] = test1_df[test1_col].fillna(default_inches)
missing_after_test1 = test1_df[test1_col].isna().sum()

# show missing counts
print(f"test 1 missing before imputation: {missing_before_test1}")
print(f"default value used (median inches): {default_inches:.2f}")
print(f"test 1 missing after imputation: {missing_after_test1}")

test 1 missing before imputation: 130
default value used (median inches): 15.60
test 1 missing after imputation: 0


In [9]:
# evaluate test 1
test1_pred_values = test1_df.loc[test1_remove_idx, test1_col]

# mae: average absolute error
test1_mae = np.mean(np.abs(test1_pred_values - test1_true_values))

# rmse: penalizes bigger errors
test1_rmse = np.sqrt(np.mean((test1_pred_values - test1_true_values) ** 2))

# close rate: percent within 0.5 inches
test1_close = (np.abs(test1_pred_values - test1_true_values) <= 0.5).mean() * 100

print("test 1 evaluation (default value imputation)")
print(f"mae: {test1_mae:.4f}")
print(f"rmse: {test1_rmse:.4f}")
print(f"within +/- 0.5 inches: {test1_close:.2f}%")

test 1 evaluation (default value imputation)
mae: 1.1046
rmse: 1.6603
within +/- 0.5 inches: 50.77%


### Imputation Test 2 - Correlation Imputation (Bivariate)

I use **correlation imputation** on `Weight` with `Ram`.
I use the correlation formula to estimate missing `Weight` values from `Ram`.

**How missing data is introduced (MAR):**
- Column: `Weight`
- Method: values are removed with higher probability when `Ram` is high.
- Missingness type: **MAR** (Missing At Random), because missingness depends on another observed column (`Ram`).

In [10]:
# test 2: mar + correlation imputation on weight
np.random.seed(222)

# copy data
test2_df = laptop_df.copy()
target_col2 = "Weight"
predictor_col2 = "Ram"

# make mar missing values based on ram
valid_rows = test2_df[test2_df[target_col2].notna() & test2_df[predictor_col2].notna()]
ram_cutoff = valid_rows[predictor_col2].median()
rand_vals = np.random.rand(len(valid_rows))
remove_mask = ((valid_rows[predictor_col2] >= ram_cutoff) & (rand_vals < 0.30)) | ((valid_rows[predictor_col2] < ram_cutoff) & (rand_vals < 0.08))
test2_remove_idx = valid_rows.index[remove_mask]
test2_true_values = test2_df.loc[test2_remove_idx, target_col2].copy()

# insert missing values
test2_df.loc[test2_remove_idx, target_col2] = np.nan
missing_before_test2 = test2_df[target_col2].isna().sum()

# calculate correlation formula parts
train = test2_df[[predictor_col2, target_col2]].dropna()
r = train[predictor_col2].corr(train[target_col2])
x_mean = train[predictor_col2].mean()
y_mean = train[target_col2].mean()
x_std = train[predictor_col2].std(ddof=0)
y_std = train[target_col2].std(ddof=0)

# fill missing values
test2_missing_mask = test2_df[target_col2].isna() & test2_df[predictor_col2].notna()
test2_df.loc[test2_missing_mask, target_col2] = y_mean + r * (y_std / x_std) * (test2_df.loc[test2_missing_mask, predictor_col2] - x_mean)
missing_after_test2 = test2_df[target_col2].isna().sum()

# show missing counts
print(f"test 2 missing before imputation: {missing_before_test2}")
print(f"correlation r (ram, weight): {r:.4f}")
print(f"test 2 missing after imputation: {missing_after_test2}")

test 2 missing before imputation: 305
correlation r (ram, weight): 0.4248
test 2 missing after imputation: 0


In [11]:
# evaluate test 2
test2_pred_values = test2_df.loc[test2_remove_idx, target_col2]

# mae: average absolute error
test2_mae = np.mean(np.abs(test2_pred_values - test2_true_values))

# rmse: penalizes bigger errors
test2_rmse = np.sqrt(np.mean((test2_pred_values - test2_true_values) ** 2))

# r^2: fit quality
test2_r2 = 1 - np.sum((test2_true_values - test2_pred_values) ** 2) / np.sum((test2_true_values - test2_true_values.mean()) ** 2)

print("test 2 evaluation (correlation imputation)")
print(f"mae: {test2_mae:.4f}")
print(f"rmse: {test2_rmse:.4f}")
print(f"r^2: {test2_r2:.4f}")

test 2 evaluation (correlation imputation)
mae: 0.4838
rmse: 0.6217
r^2: 0.0058


### Imputation Test 3 - Similarity-based Imputation (Multivariate)

I use **similarity-based imputation** on `Price_euros`.
Each missing price is replaced by the average price of the most similar laptops.

**How missing data is introduced (MNAR):**
- Column: `Price_euros`
- Method: higher-priced laptops are removed with higher probability.
- Missingness type: **MNAR** (Missing Not At Random), because missingness depends on the price value itself.

In [12]:
# test 3: mnar + similarity-based imputation on price_euros
np.random.seed(333)

# copy data
test3_df = laptop_df.copy()
target_col3 = "Price_euros"
feature_cols3 = ["Ram", "Weight", "Inches"]

# make mnar missing values based on price
valid_target = test3_df[test3_df[target_col3].notna()]
price_q75 = valid_target[target_col3].quantile(0.75)
rand_vals = np.random.rand(len(valid_target))
remove_mask = ((valid_target[target_col3] > price_q75) & (rand_vals < 0.35)) | ((valid_target[target_col3] <= price_q75) & (rand_vals < 0.08))
test3_remove_idx = valid_target.index[remove_mask]
test3_true_values = test3_df.loc[test3_remove_idx, target_col3].copy()

# insert missing values
test3_df.loc[test3_remove_idx, target_col3] = np.nan
missing_before_test3 = test3_df[target_col3].isna().sum()

# prepare rows with known prices
known_rows = test3_df[test3_df[target_col3].notna()].copy()
known_features = known_rows[feature_cols3].copy()
feature_means = known_features.mean()
known_features = known_features.fillna(feature_means)
missing_idx = test3_df.index[test3_df[target_col3].isna()]

# fill each missing price using 5 most similar rows
k = 5
for idx in missing_idx:
    row_features = test3_df.loc[idx, feature_cols3].fillna(feature_means)
    score = (known_features["Ram"] - row_features["Ram"]).abs() + (known_features["Weight"] - row_features["Weight"]).abs() + (known_features["Inches"] - row_features["Inches"]).abs()
    nearest_idx = score.nsmallest(k).index
    test3_df.loc[idx, target_col3] = known_rows.loc[nearest_idx, target_col3].mean()

missing_after_test3 = test3_df[target_col3].isna().sum()

# show missing counts
print(f"test 3 missing before imputation: {missing_before_test3}")
print(f"test 3 missing after imputation: {missing_after_test3}")

test 3 missing before imputation: 208
test 3 missing after imputation: 0


In [13]:
# evaluate test 3
test3_pred_values = test3_df.loc[test3_remove_idx, target_col3]

# mae: average absolute error
test3_mae = np.mean(np.abs(test3_pred_values - test3_true_values))

# rmse: penalizes bigger errors
test3_rmse = np.sqrt(np.mean((test3_pred_values - test3_true_values) ** 2))

# r^2: fit quality
test3_r2 = 1 - np.sum((test3_true_values - test3_pred_values) ** 2) / np.sum((test3_true_values - test3_true_values.mean()) ** 2)

print("test 3 evaluation (similarity-based imputation)")
print(f"mae: {test3_mae:.4f}")
print(f"rmse: {test3_rmse:.4f}")
print(f"r^2: {test3_r2:.4f}")

test 3 evaluation (similarity-based imputation)
mae: 361.2186
rmse: 543.4278
r^2: 0.5513
